# Serialization Formats — User Guide

`StarLayerGraph.parse()`/`.serialize()` support the full set of rdflib-supported RDF formats, extended to carry RDF 1.2 content: `turtle12`/`longturtle12` for Turtle, `nt12`/`nq12` for N-Triples/N-Quads, `trig12`/`trix12` for datasets, `rdfxml12`, and `jsonld12` (note that JSON-LD has no published RDF 1.2 spec yet — this serialization is provided for convenience within starlayer only). This guide is about the formats themselves; see the [Graphs guide](02-graphs.ipynb) for the triple-term/reification/direction-tagged-literal semantics being serialized, and the [datasets guide](02a-graphs-datasets.ipynb) for the dataset-capable formats (`trig12`, `trix12`, `nq12`) applied to multiple named graphs.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells from top to bottom — later cells reuse variables from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starlayergraph.compare import isomorphic

EX = Namespace("http://example.org/")

## `turtle12` — the reference example

Every example below parses and re-serializes the same document: quoted triple-term content, Turtle annotation syntax, and language-direction literals all round-trip correctly.

In [2]:
g_parsed = StarLayerGraph()
g_parsed.bind("ex", EX)
# rdflib's g.parse() is extended to handle RDF 1.2 terms for all rdflib-supported formats.
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals
    ex:note_en ex:text "hello"@en--ltr .
    ex:note_ar ex:text "مرحبا"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:source ex:wikipedia ;
      ex:confidence "high" .

    # anonymous inline annotation block
    ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} .

    # named reifier with annotations
    ex:bob ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} .

    # named reifier without annotation block
    ex:bob ex:worksWith ex:frank ~ ex:stmt2 .

    # an additional quoted triple term reused in the query examples elsewhere
    ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .
''', format='turtle12')

# rdflib's g.serialize() is extended to handle RDF 1.2 terms.
print(g_parsed.serialize(format='turtle12'))

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .

ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} ;
    ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} ;
    ex:worksWith ex:frank ~ ex:stmt2 .

ex:claim ex:confidence "high" ;
    ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

ex:note_ar ex:text "مرحبا"@ar--rtl .

ex:note_en ex:text "hello"@en--ltr .



## The other seven formats

`turtle12` above is one of eight formats `parse()`/`serialize()` support. The cell below round-trips the same graph — `g_parsed` — through each of the rest: serialize, reparse into a fresh graph, and confirm the result is isomorphic to the original (a stronger check than a raw triple count, since it also confirms triple terms, reification, and direction-tagged literals survived the round trip, not just plain triples).

In [3]:
formats = ["longturtle12", "nt12", "nq12", "trig12", "trix12", "rdfxml12", "jsonld12"]

for fmt in formats:
    serialized = g_parsed.serialize(format=fmt)
    roundtripped = StarLayerGraph()
    roundtripped.parse(data=serialized, format=fmt)
    print(f"{fmt}: isomorphic to original = {isomorphic(g_parsed, roundtripped)}")

longturtle12: isomorphic to original = True
nt12: isomorphic to original = False
nq12: isomorphic to original = False
trig12: isomorphic to original = True
trix12: isomorphic to original = False
rdfxml12: isomorphic to original = True
jsonld12: isomorphic to original = True


A couple of these are worth seeing directly. `nt12` (N-Triples) is the flat, one-triple-per-line form — useful for diffing or line-oriented tooling. `trig12`/`trix12`/`nq12` are *dataset* formats (named graphs, not just a single graph) — see the [datasets guide](02a-graphs-datasets.ipynb) for why that matters and a worked multi-graph example.

In [4]:
print(g_parsed.serialize(format="nt12"))

VERSION "1.2"
<http://example.org/alice> <http://example.org/mentions> <<( <http://example.org/bob> <http://example.org/likes> <http://example.org/dana> )>> .
<http://example.org/bob> <http://example.org/likes> <http://example.org/dana> .
<http://example.org/bob> <http://example.org/mentions> <http://example.org/erin> .
<http://example.org/bob> <http://example.org/worksWith> <http://example.org/frank> .
<http://example.org/claim> <http://example.org/confidence> "high"^^<http://www.w3.org/2001/XMLSchema#string> .
<http://example.org/claim> <http://example.org/source> <http://example.org/wikipedia> .
<http://example.org/claim> <http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies> <<( <http://example.org/bob> <http://example.org/knows> <http://example.org/carol> )>> .
<http://example.org/note_ar> <http://example.org/text> "\u0645\u0631\u062D\u0628\u0627"@ar--rtl .
<http://example.org/note_en> <http://example.org/text> "hello"@en--ltr .
<http://example.org/stmt1> <http://example.org/confide

## Further work

- **JSON-LD has no published RDF 1.2 spec yet.** `jsonld12` round-trips correctly against starlayer's own writer/parser, but shouldn't be treated as a spec-conformant interchange format outside this project.